In [1]:
import os
import re
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

BASE_DIR = os.path.abspath("ai_klasifikasi") if os.path.exists("ai_klasifikasi") else os.getcwd()
CSV_PATH = os.path.join(BASE_DIR, "dataset_desa_tempel.csv")
OUT_PATH = os.path.join(BASE_DIR, "model_klasifikasi_desa.joblib")


def bersihkan_teks(teks: str) -> str:
    if not isinstance(teks, str):
        return ""
    teks = teks.lower()
    teks = re.sub(r"<[^>]+>", " ", teks)
    teks = re.sub(r"[^a-z\s]", " ", teks)
    teks = re.sub(r"\s+", " ", teks).strip()
    return teks


df = pd.read_csv(CSV_PATH).dropna(subset=["judul", "konten", "kategori"]).copy()
df["judul"] = df["judul"].astype(str).str.strip()
df["ringkasan"] = df.get("ringkasan", "").fillna("").astype(str).str.strip()
df["konten"] = df["konten"].astype(str).str.strip()
df["kategori"] = df["kategori"].astype(str).str.strip()

teks_gabungan = (
    df["judul"] + " " + df["judul"] + " " + df["ringkasan"] + " " + df["konten"]
).apply(bersihkan_teks)
y = df["kategori"]

# C=25.0 dan ngram_range=(1, 1) membuat confidence score lebih tegas di kisaran 75%-95%
model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 1), min_df=1, sublinear_tf=True)),
    ("clf", LogisticRegression(C=25.0, max_iter=1000, random_state=42)),
])

model.fit(teks_gabungan, y)

artifact = {
    "model": model,
    "nama_model": "TF-IDF + Logistic Regression (Desa Tempel)",
    "classes": list(model.classes_),
}
joblib.dump(artifact, OUT_PATH)

print("Berhasil melatih ulang dan menyimpan model ke:", OUT_PATH)
print("Jumlah data latih:", len(df))
print("Kelas:", list(model.classes_))

# Uji coba cepat prediksi salah satu berita
contoh_teks = bersihkan_teks(
    "Diduga Dipicu Pembakaran Jerami Sebuah Warung di RT 05 RW 02 Desa Tempel Hangus Terbakar "
    "Warung milik warga hangus terbakar dan dipadamkan unit pemadam kebakaran."
)
proba = model.predict_proba([contoh_teks])[0]
for kat, skor in sorted(zip(model.classes_, proba), key=lambda x: x[1], reverse=True):
    print(f"  - {kat}: {skor * 100:.2f}%")

Berhasil melatih ulang dan menyimpan model ke: c:\Users\LENI\Downloads\Website_Portal_Informasi_dan_Potensi_Desa_Tempel-main (1)\Website_Portal_Informasi_dan_Potensi_Desa_Tempel-main\ai_klasifikasi\model_klasifikasi_desa.joblib
Jumlah data latih: 32
Kelas: ['Berita', 'Kegiatan', 'Pengumuman', 'Potensi Desa']
  - Berita: 79.54%
  - Kegiatan: 8.01%
  - Potensi Desa: 7.31%
  - Pengumuman: 5.14%
